In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input 
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
df = pd.read_csv(r"C:\Users\DELL\OneDrive\Documents\Machine Learning\loan.csv")
df.head(5)

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [3]:
df.shape

(614, 13)

In [4]:
df.isnull().sum()

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

In [5]:
df.dtypes

Loan_ID               object
Gender                object
Married               object
Dependents            object
Education             object
Self_Employed         object
ApplicantIncome        int64
CoapplicantIncome    float64
LoanAmount           float64
Loan_Amount_Term     float64
Credit_History       float64
Property_Area         object
Loan_Status           object
dtype: object

In [6]:
df.dropna(inplace=True)

In [7]:
df.drop(columns=["Loan_ID"], axis=1, inplace=True)

In [8]:
from sklearn.preprocessing import LabelEncoder
lab = LabelEncoder()

In [22]:
df["Gender"] = lab.fit_transform(df["Gender"])
df["Married"] = lab.fit_transform(df["Married"])
df["Dependents"] = lab.fit_transform(df["Dependents"])
df["Education"] = lab.fit_transform(df["Education"])
df["Self_Employed"] = lab.fit_transform(df["Self_Employed"])
df["Loan_Status"] = lab.fit_transform(df["Loan_Status"])

In [23]:
df.describe()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status
count,480.000000,480.000000,480.000000,480.000000,480.000000,4.800000e+02,4.800000e+02,4.800000e+02,4.800000e+02,480.000000,480.000000
mean,0.820833,0.647917,0.777083,0.202083,0.137500,2.590520e-17,1.572816e-17,-1.554312e-16,-2.146431e-16,0.854167,0.691667
std,0.383892,0.478118,1.020815,0.401973,0.344734,1.001043e+00,1.001043e+00,1.001043e+00,1.001043e+00,0.353307,0.462287
min,0.000000,0.000000,0.000000,0.000000,0.000000,-9.208610e-01,-6.046330e-01,-1.687742e+00,-4.698022e+00,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000,-4.354171e-01,-6.046330e-01,-5.562428e-01,2.755416e-01,1.000000,0.000000
50%,1.000000,1.000000,0.000000,0.000000,0.000000,-2.658318e-01,-1.899046e-01,-2.080892e-01,2.755416e-01,1.000000,1.000000
75%,1.000000,1.000000,2.000000,0.000000,0.000000,8.623086e-02,2.570423e-01,3.141413e-01,2.755416e-01,1.000000,1.000000
max,1.000000,1.000000,3.000000,1.000000,1.000000,1.335768e+01,1.233512e+01,5.660787e+00,2.117602e+00,1.000000,1.000000


In [24]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()

In [25]:
df["ApplicantIncome"] = sc.fit_transform(df[["ApplicantIncome"]])
df["CoapplicantIncome"] = sc.fit_transform(df[["CoapplicantIncome"]])
df["LoanAmount"] = sc.fit_transform(df[["LoanAmount"]])
df["Loan_Amount_Term"] = sc.fit_transform(df[["Loan_Amount_Term"]])

In [26]:
df["Loan_Status"].value_counts()

Loan_Status
1    332
0    148
Name: count, dtype: int64

In [27]:
df.drop(columns=["Property_Area"], inplace=True)

KeyError: "['Property_Area'] not found in axis"

In [28]:
x = df.drop("Loan_Status", axis=1)
y = df["Loan_Status"]

In [29]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [47]:
from imblearn.over_sampling import SMOTE
sm = SMOTE()
X_train, y_train = sm.fit_resample(X_train, y_train)

In [48]:
model = Sequential([Input(shape=[X_train.shape[1],]), 
                    Dense(30, activation='relu'),
                    Dropout(0.2),
                    Dense(15, activation='relu'),
                    Dense(1, activation='sigmoid')
                    ])

In [49]:
model.compile(optimizer='sgd', loss='binary_crossentropy', metrics=["accuracy"])

In [50]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [51]:
model.fit(X_train, y_train, epochs=30, batch_size=15, callbacks=early_stopping, verbose=1)

Epoch 1/30


36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5833 - loss: 0.6856 
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5720 - loss: 0.6858
Epoch 3/30
 1/36 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.4000 - loss: 0.7266

c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5852 - loss: 0.6827
Epoch 4/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5852 - loss: 0.6863
Epoch 5/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6117 - loss: 0.6735
Epoch 6/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6231 - loss: 0.6738
Epoch 7/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6345 - loss: 0.6711
Epoch 8/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6231 - loss: 0.6647
Epoch 9/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6591 - loss: 0.6633
Epoch 10/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6477 - loss: 0.6583
Epoch 11/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6705 - loss: 0.6624
Epoch 12/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6477 - loss: 0.6565
Epoch 13/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6667 - loss: 0.6485
Epoch 14/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6723 - loss: 0.649

In [52]:
y_prob= model.predict(X_test)
y_pred = (y_prob > 0.5).astype(int).ravel()

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step


In [53]:
print("Accuracy Score", accuracy_score(y_test, y_pred))


Accuracy Score 0.7916666666666666


In [54]:
print("Classification Report:", classification_report(y_test, y_pred))

Classification Report:               precision    recall  f1-score   support

           0       0.70      0.50      0.58        28
           1       0.82      0.91      0.86        68

    accuracy                           0.79        96
   macro avg       0.76      0.71      0.72        96
weighted avg       0.78      0.79      0.78        96

